In [18]:
import json
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np

# ----- Keys -----
STRUCTURAL_KEYS = ['Qubits', 'Qumodes', 'Qubit Gates', 'Qumode Gates', 'Hybrid Gates', 'Circuit Depth']
STRUCTURAL_LABELS = ['Qubits', 'Qumodes', 'Qubit Gates', 'Qumode\nGates', 'Hybrid Gates', 'Circuit Depth']

PERFORMANCE_KEYS = ['Truncation Cost', 'Wigner Negativity', 'Average Energy']
PERFORMANCE_LABELS = ['Truncation\nCost', 'Wigner Negativity', 'Average Energy']

# ----- Load Metrics -----
def load_metrics_from_json(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    struct_all = {}
    perf_all = {}
    for name, metrics in data.items():
        struct_all[name] = {k: metrics["structural"].get(k, 0.0) for k in STRUCTURAL_KEYS}
        perf_all[name] = {k: metrics["quantum"].get(k, 0.0) for k in PERFORMANCE_KEYS}
    
    return struct_all, perf_all

In [3]:
def plot_radar_group(metrics_dict, keys, display_labels, filename, show_values=True):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Circle
    import numpy as np

    # Compute max for each metric
    max_per_key = {
        k: max(metrics.get(k, 0) for metrics in metrics_dict.values())
        for k in keys
    }

    # Normalize metrics
    normalized_metrics = {
        label: {
            k: (metrics.get(k, 0) / max_per_key[k] if max_per_key[k] != 0 else 0)
            for k in keys
        }
        for label, metrics in metrics_dict.items()
    }

    num_plots = len(metrics_dict)
    rows, cols = 2, (num_plots + 1) // 2
    fig, axs = plt.subplots(rows, cols, subplot_kw=dict(polar=True),
                            figsize=(4.2 * cols, 4.2 * rows))
    fig.patch.set_facecolor("white")
    axs = axs.flatten()

    for idx, (label, _) in enumerate(metrics_dict.items()):
        ax = axs[idx]

        N = len(keys)
        angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
        angles += angles[:1]

        vals = [normalized_metrics[label][k] for k in keys]
        vals += vals[:1]

        color = plt.cm.tab10(idx % 10)

        ax.set_facecolor((*color[:3], 0.07))
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(display_labels, fontsize=13, color="#222222", fontweight="medium")

        ax.set_yticks([])
        ax.set_ylim(0, 1.2)
        ax.grid(color="gray", linestyle="--", linewidth=0.5)
        ax.spines["polar"].set_visible(False)

        for radius in np.linspace(0.2, 1.0, 4) * 1.2:
            ax.add_patch(Circle((0, 0), radius, transform=ax.transData._b,
                                color=color, alpha=0.08, zorder=0))

        ax.plot(angles, vals, linewidth=1.8, color=color)
        ax.fill(angles, vals, color=color, alpha=0.25)

        if show_values:
            fixed_radius = 1.05
            offset_angle_tol = 0.1  # radians tolerance (~5.7°)

            for j in range(N):
                angle = angles[j]
                r = fixed_radius
                va = "center"

                # For 0 degrees (right), push values down and align top (so values below labels)
                if abs(angle) < offset_angle_tol or abs(angle - 2 * np.pi) < offset_angle_tol:
                    r -= 0.25
                    va = "top"

                # For 180 degrees (left), push values up and align top (so values below labels)
                elif abs(angle - np.pi) < offset_angle_tol:
                    r -= 0.2
                    va = "top"

                ax.text(angle, r, f"{vals[j]:.2f}",
                        ha="center", va=va,
                        fontsize=9, color="#111111", fontweight="semibold")

        ax.set_title(label, fontsize=16, pad=20, color=color, weight="bold")

    for i in range(len(metrics_dict), len(axs)):
        fig.delaxes(axs[i])

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()


In [16]:
def plot_radar_group(metrics_dict, keys, display_labels, filename, show_values=True):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Circle
    import numpy as np

    # Compute max for each metric
    max_per_key = {
        k: max(metrics.get(k, 0) for metrics in metrics_dict.values())
        for k in keys
    }

    # Normalize metrics
    normalized_metrics = {
        label: {
            k: (metrics.get(k, 0) / max_per_key[k] if max_per_key[k] != 0 else 0)
            for k in keys
        }
        for label, metrics in metrics_dict.items()
    }

    def polygon_area(angles, values):
        angles = np.array(angles[:-1])
        values = np.array(values[:-1])
        x = values * np.cos(angles)
        y = values * np.sin(angles)
        return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

    num_plots = len(metrics_dict)
    rows, cols = 2, (num_plots + 1) // 2
    fig, axs = plt.subplots(rows, cols, subplot_kw=dict(polar=True),
                            figsize=(4.8 * cols, 4.8 * rows))
    fig.patch.set_facecolor("white")
    axs = axs.flatten()

    for idx, (label, _) in enumerate(metrics_dict.items()):
        ax = axs[idx]

        N = len(keys)
        angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
        angles += angles[:1]

        vals = [normalized_metrics[label][k] for k in keys]
        vals += vals[:1]

        color = plt.cm.tab10(idx % 10)

        ax.set_facecolor((*color[:3], 0.07))
        ax.set_xticks([])  # Hide default tick labels
        ax.set_yticks([])
        ax.set_ylim(0, 1.2)
        ax.grid(color="gray", linestyle="--", linewidth=0.5)
        ax.spines["polar"].set_visible(False)

        # Place custom labels further outward
        label_radius = 1.35
        for angle, label_text in zip(angles[:-1], display_labels):
            ax.text(angle, label_radius, label_text,
                    fontsize=16, fontweight="bold",
                    color="#222222", ha="center", va="center")

        for radius in np.linspace(0.2, 1.0, 4) * 1.2:
            ax.add_patch(Circle((0, 0), radius, transform=ax.transData._b,
                                color=color, alpha=0.08, zorder=0))

        ax.plot(angles, vals, linewidth=2.0, color=color)
        ax.fill(angles, vals, color=color, alpha=0.25)

        if show_values:
            fixed_radius = 1.05
            offset_angle_tol = 0.5

            for j in range(N):
                angle = angles[j]
                r = fixed_radius
                va = "center"

                if abs(angle) < offset_angle_tol or abs(angle - 2 * np.pi) < offset_angle_tol:
                    r -= 0.25
                    va = "top"
                elif abs(angle - np.pi) < offset_angle_tol:
                    r -= 0.25
                    va = "top"
                elif abs(angle - np.pi/2) < offset_angle_tol:
                    r -= 0.8  # push further down for top labels
                    va = "top"
                elif abs(angle - 3*np.pi/2) < offset_angle_tol:
                    r += 0.1
                    va = "bottom"

                ax.text(angle, r, f"{vals[j]:.2f}",
                        ha="center", va=va,
                        fontsize=13, color="#111111", fontweight="bold")

        area = polygon_area(angles, vals)
        ax.set_title(f"{label}  (Area: {area:.3f})",
                     fontsize=18, pad=22, color=color, weight="bold")

    for i in range(len(metrics_dict), len(axs)):
        fig.delaxes(axs[i])

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()




In [19]:
json_path = "benchmark_metrics.json"  # adjust if needed
struct_all, perf_all = load_metrics_from_json(json_path)

SCRIPT_DIR = os.getcwd()  # Works in Jupyter
OUTPUT_DIR = os.path.join(SCRIPT_DIR, "circuit_characters")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Plot structural metrics
plot_radar_group(struct_all, STRUCTURAL_KEYS, STRUCTURAL_LABELS, os.path.join(OUTPUT_DIR, "summary_structural.png"),show_values=True)
plot_radar_group(perf_all, PERFORMANCE_KEYS, PERFORMANCE_LABELS,os.path.join(OUTPUT_DIR, "summary_quantum.png"),show_values = True)